In [ ]:
# Binary Thresholding
# Separating the crack pixels from the road. 
# This creates a clear mask where the defects are isolated from the background.
_, crack_mask = cv2.threshold(final_enhanced, 35, 255, cv2.THRESH_BINARY_INV)


# We classify images into 3 categories based on their Variation Ratio to apply the most suitable cleaning and connecting filters.

if ratio > 0.6: 
    # Extremely High Contrast (Clear Images)
    # The crack is already prominent, so we avoid 'Opening' to prevent losing detail.
    print("High Contrast. Skipping Opening phase.")
    
    # For visualization consistency, we map 'cleaned' to the original mask.
    cleaned = crack_mask.copy() 
    
    # Using a small 3x3 kernel for a gentle 'Closing' to bridge tiny gaps.
    kernel_cl = np.ones((3,3), np.uint8)
    connected = cv2.morphologyEx(crack_mask, cv2.MORPH_CLOSE, kernel_cl, iterations=1)
    
elif 0.4 < ratio <= 0.6: 
    # Medium Contrast (Mild Noise/Texture)
    # Requires a balance between cleaning and detail preservation.
    print(" Medium Contrast. Applying light cleaning.")
    
    # 'Opening' with a tiny 2x2 kernel to remove micro-noise without erasing fine cracks.
    kernel_small = np.ones((2,2), np.uint8) 
    cleaned = cv2.morphologyEx(crack_mask, cv2.MORPH_OPEN, kernel_small, iterations=1)
    
    # 'Closing' with a 5x5 kernel to strengthen the crack structure.
    kernel_cl = np.ones((3,3), np.uint8)
    connected = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel_cl, iterations=2)
    
else: 
    # Standard/Low Contrast (Fragmented or Noisy Images)
    # Cracks are likely broken into pixels; needs aggressive cleaning and merging.
    print("Standard/Low Contrast. Applying full cleaning.")
    
    # Standard 3x3 'Opening' to eliminate significant road surface artifacts.
    kernel = np.ones((3,3), np.uint8)
    cleaned = cv2.morphologyEx(crack_mask, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # Strong 9x9 'Closing' to bridge large gaps and merge fragmented red-dots into lines.
    kernel_cl = np.ones((9,9), np.uint8)
    connected = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel_cl, iterations=2)
